# P2 GPU validation (Task 3.6B) — execution wrapper only

Runs the committed P2 validation harness on CUDA. This notebook makes
**no scientific decisions**: protocol, gate, checkpoint, and seeds are
pinned below and verified before/after execution.

Fail-closed rules: CUDA missing → STOP · wrong commit → STOP ·
checkpoint mismatch → STOP · missing input → recorded omission.
Never paste a GitHub token here; results leave via file download.

## Cell 1 — environment identity (STOP if CUDA is unavailable)

In [ ]:
import platform
import torch
print('python:', platform.python_version())
print('torch:', torch.__version__, '| cuda_available:',
      torch.cuda.is_available())
assert torch.cuda.is_available(), \
    'STOP: GPU validation unresolved: CUDA unavailable'
print('gpu:', torch.cuda.get_device_name(0))
print('gpu count:', torch.cuda.device_count())
print('cuda version:', torch.version.cuda)
import mace, ase
print('mace:', mace.__version__, '| ase:', ase.__version__)
import numpy
print('numpy:', numpy.__version__)

## Cell 2 — obtain exact repository state (STOP on mismatch)

In [ ]:
cd /kaggle/working && git clone https://github.com/wt2018mask/Rhombus.git Rhombus
cd /kaggle/working/Rhombus && git checkout 44029b861044cdc615dcabbdb5f1a77e20b5bc8d
test "$(git rev-parse HEAD)" = "44029b861044cdc615dcabbdb5f1a77e20b5bc8d" || (echo 'STOP: wrong commit'; exit 1)
test -z "$(git status --porcelain)" || (echo 'STOP: dirty tree'; exit 1)
echo "commit verified: $(git rev-parse HEAD)"

## Cell 3 — dependencies (then re-verify CUDA survived pip)

In [ ]:
cd /kaggle/working/Rhombus && pip install -q -r requirements.txt
pip freeze | grep -iE '^(torch|mace-torch|ase|numpy|scipy|pymatgen) '
python -c "import torch; assert torch.cuda.is_available(), 'STOP: torch install lost CUDA'; print('cuda still available:', torch.cuda.get_device_name(0))"

## Cell 4 — checkpoint identity (STOP on mismatch)

In [ ]:
cd /kaggle/working/Rhombus && python -c "
import yaml
from rudeus.mlip.relax import ensure_checkpoint, default_model_path
cfg = yaml.safe_load(open('config.yaml', encoding='utf-8'))['mlip']
p = ensure_checkpoint(cfg['checkpoint_url'], default_model_path(),
                      cfg['checkpoint_sha256'])
assert cfg['checkpoint_sha256'] == '75428afe3a1d7d8062e19bcaabd5c433623cabf308242ec9fb493e38604fb638', 'STOP: pin drift'
print('checkpoint verified:', p)
"

## Cell 5 — input verification dry-run (no MD; 2 omissions expected)

In [ ]:
cd /kaggle/working/Rhombus && python -m rudeus.mlip.validation --cases control-1e9,0d4de6bc,f8857e1e,4a9735f2,628136c9 --control-input data/batches/audit/p2_validation_control_1e9.json
# EXPECTED: control-1e9 OK, both marginals OK, both collapse cases
# OMITTED (no P1 relaxed records - recorded, never substituted).
# Proceed only if control + both marginals resolve.

## Cell 6 — validation runs, invocation A (5 MD max + 2 omissions)

In [ ]:
cd /kaggle/working/Rhombus && python -m rudeus.mlip.validation --run --cases control-1e9,0d4de6bc,f8857e1e,4a9735f2,628136c9 --control-input data/batches/audit/p2_validation_control_1e9.json --records-dir /kaggle/working/p2val --report /kaggle/working/p2_validation.json --device cuda --worker kaggle-gpu --seed-base 20260913 --run-index-base 0

## Cell 7 — validation runs, invocation B (marginals, second seeds)

In [ ]:
cd /kaggle/working/Rhombus && python -m rudeus.mlip.validation --run --cases 0d4de6bc,f8857e1e --records-dir /kaggle/working/p2val --report /kaggle/working/p2_validation.json --device cuda --worker kaggle-gpu --seed-base 20260914 --run-index-base 1  # rewrites report covering all 7 runs

## Cell 8 — output verification (asserts, not edits)

In [ ]:
import glob, json
rep = json.load(open('/kaggle/working/p2_validation.json'))
env = rep['environment']
assert env['device'] == 'cuda', env
assert env['cuda_available'] is True
assert env['gpu_name'], 'GPU identity must be recorded'
assert rep['git_commit'] == '44029b861044cdc615dcabbdb5f1a77e20b5bc8d', rep['git_commit']
assert rep['checkpoint']['sha256'] == '75428afe3a1d7d8062e19bcaabd5c433623cabf308242ec9fb493e38604fb638'
assert rep['protocol_hash'] == 'eccec569eca0e2a9', rep['protocol_hash']
assert glob.glob('/kaggle/working/p2val/*.tmp') == []  # no partials
for r in rep['runs']:
    assert r['dynamic_state'] in ('PASS', 'FAIL', 'INDETERMINATE'), r
    print(f"{r['case_id']:>16} seed={r['seed']} completed={r['completion']} state={r['dynamic_state']} "
          f"Lind={r['lindemann']} RMSD={r['host_rmsd_final']}")
print('missing cases:', rep['summary'].get('cases_missing', []))
print('gpu status:', rep['gpu_validation_status'])

## Cell 9 — handoff (download, do NOT push from here)

1. Download `/kaggle/working/p2_validation.json` (+ `p2val/` records)
   via the Kaggle UI file browser.
2. Commit locally next to the run (never paste tokens into this notebook).
3. Execution manifest (also printed by cell 8):
   repo=https://github.com/wt2018mask/Rhombus.git commit=44029b861044cdc615dcabbdb5f1a77e20b5bc8d seeds=20260913/20260914 protocol=eccec569eca0e2a9 checkpoint=75428afe3a1d…